We're:

Creating a SQLite database (lightweight database stored as a file)
Loading our cleaned CSV data into a table
Creating a connection so we can run SQL queries

SQLite is perfect for projects like this — it's built into Python, requires no setup, and stores everything in one file.

In [1]:
import pandas as pd
import sqlite3
import os

# ── SETUP PATHS ──────────────────────────────────────────────────────────
os.chdir("/Users/animesh/Documents/personal_git/melbourne-crime-liveability-dashboard")
PROCESSED = "data/processed/"
SQL_OUTPUT = "sql/"
os.makedirs(SQL_OUTPUT, exist_ok=True)

print("📊 PHASE 2: SQL ANALYSIS")
print("="*60)
print("\nStep 1: Creating SQLite database...\n")

# ── CREATE DATABASE CONNECTION ────────────────────────────────────────────
# This creates a file called 'crime_data.db' in our project
db_path = "crime_data.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"✓ Database created: {db_path}")

# ── LOAD CLEANED CSV INTO DATABASE ────────────────────────────────────────
print("\nStep 2: Loading cleaned data into database table...\n")

# Read the CSV we created in Phase 1
crime_df = pd.read_csv(PROCESSED + "crime_clean.csv")

print(f"✓ Loaded {len(crime_df)} rows from CSV")
print(f"✓ Columns: {crime_df.columns.tolist()}")

# Write the data to SQLite as a table called 'crimes'
# if_exists='replace' means: if table already exists, replace it
crime_df.to_sql('crimes', conn, if_exists='replace', index=False)

print(f"✓ Created table 'crimes' in database")

# ── VERIFY TABLE WAS CREATED ──────────────────────────────────────────────
print("\nStep 3: Verifying table...\n")

# Check how many rows in the table
cursor.execute("SELECT COUNT(*) FROM crimes")
row_count = cursor.fetchone()[0]
print(f"✓ Total rows in 'crimes' table: {row_count}")

# Show the structure of the table
cursor.execute("PRAGMA table_info(crimes)")
columns = cursor.fetchall()
print(f"✓ Table columns:")
for col in columns:
    col_name, col_type = col[1], col[2]
    print(f"   - {col_name}: {col_type}")

# Show first few rows
print(f"\n✓ First 5 rows of data:")
cursor.execute("SELECT * FROM crimes LIMIT 5")
rows = cursor.fetchall()
for row in rows:
    print(f"   {row}")

print(f"\n✅ Database setup complete!")
print(f"✅ Ready to run SQL queries!")

📊 PHASE 2: SQL ANALYSIS

Step 1: Creating SQLite database...

✓ Database created: crime_data.db

Step 2: Loading cleaned data into database table...

✓ Loaded 290 rows from CSV
✓ Columns: ['year', 'year_ending', 'police_region', 'lga_name', 'incidents', 'rate_per_100k']
✓ Created table 'crimes' in database

Step 3: Verifying table...

✓ Total rows in 'crimes' table: 290
✓ Table columns:
   - year: INTEGER
   - year_ending: TEXT
   - police_region: TEXT
   - lga_name: TEXT
   - incidents: INTEGER
   - rate_per_100k: REAL

✓ First 5 rows of data:
   (2024, 'December', '1 North West Metro', 'BANYULE', 7590, 5702.17011683)
   (2024, 'December', '1 North West Metro', 'BRIMBANK', 14119, 7124.608799554)
   (2024, 'December', '1 North West Metro', 'DAREBIN', 13982, 8771.785831693)
   (2024, 'December', '1 North West Metro', 'HOBSONS BAY', 5973, 6293.377882117)
   (2024, 'December', '1 North West Metro', 'HUME', 16427, 6074.756225679)

✅ Database setup complete!
✅ Ready to run SQL queries!


What We're Doing:

We're finding how many incidents each LGA had each year (2015-2024). This shows crime trends over time.

In [2]:
print("\n" + "="*60)
print("QUERY 1: CRIME TREND BY LGA AND YEAR")
print("="*60)
print("\nSQL Query:")
print("""
SELECT 
    year,
    lga_name,
    SUM(incidents) as total_incidents,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate
FROM crimes
GROUP BY year, lga_name
ORDER BY year DESC, total_incidents DESC
""")

print("\nWhat this does:")
print("- SUM(incidents): Adds up all incidents for that LGA in that year")
print("- AVG(rate_per_100k): Calculates average crime rate")
print("- GROUP BY: Groups results by year and LGA")
print("- ORDER BY: Sorts by newest year first, then by most incidents\n")

# ── RUN THE QUERY ─────────────────────────────────────────────────────────
query1 = """
SELECT 
    year,
    lga_name,
    SUM(incidents) as total_incidents,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate
FROM crimes
GROUP BY year, lga_name
ORDER BY year DESC, total_incidents DESC
"""

df_query1 = pd.read_sql_query(query1, conn)

print("Results Preview (first 20 rows):")
print(df_query1.head(20))

print(f"\nTotal rows returned: {len(df_query1)}")

# Export to CSV
output_file1 = SQL_OUTPUT + "01_trend_by_lga_year.csv"
df_query1.to_csv(output_file1, index=False)
print(f"\n✓ Exported to: {output_file1}")


QUERY 1: CRIME TREND BY LGA AND YEAR

SQL Query:

SELECT 
    year,
    lga_name,
    SUM(incidents) as total_incidents,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate
FROM crimes
GROUP BY year, lga_name
ORDER BY year DESC, total_incidents DESC


What this does:
- SUM(incidents): Adds up all incidents for that LGA in that year
- AVG(rate_per_100k): Calculates average crime rate
- GROUP BY: Groups results by year and LGA
- ORDER BY: Sorts by newest year first, then by most incidents

Results Preview (first 20 rows):
    year              lga_name  total_incidents  avg_crime_rate
0   2024             MELBOURNE            32840        16965.34
1   2024                 CASEY            20421         5071.47
2   2024               WYNDHAM            17819         5294.88
3   2024                  HUME            16427         6074.76
4   2024     GREATER DANDENONG            16171         9704.40
5   2024              BRIMBANK            14119         7124.61
6   2024               DAR

What We're Doing:

We're finding which LGAs have the MOST crime incidents (ranking them 1-10). This shows which suburbs are most dangerous.

In [3]:
print("\n" + "="*60)
print("QUERY 2: TOP 10 HIGHEST CRIME LGAs")
print("="*60)
print("\nSQL Query:")
print("""
SELECT 
    lga_name,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT year) as years_of_data,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MAX(rate_per_100k), 2) as max_crime_rate
FROM crimes
GROUP BY lga_name
ORDER BY total_incidents DESC
LIMIT 10
""")

print("\nWhat this does:")
print("- Calculates TOTAL incidents for each LGA (summed across all years)")
print("- Shows how many years of data we have")
print("- Shows average and maximum crime rates")
print("- LIMIT 10: Only show top 10 LGAs\n")

# ── RUN THE QUERY ─────────────────────────────────────────────────────────
query2 = """
SELECT 
    lga_name,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT year) as years_of_data,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MAX(rate_per_100k), 2) as max_crime_rate
FROM crimes
GROUP BY lga_name
ORDER BY total_incidents DESC
LIMIT 10
"""

df_query2 = pd.read_sql_query(query2, conn)

print("Results (Top 10 Highest Crime LGAs):")
print(df_query2.to_string(index=False))

# Export to CSV
output_file2 = SQL_OUTPUT + "02_top10_highest_crime.csv"
df_query2.to_csv(output_file2, index=False)
print(f"\n✓ Exported to: {output_file2}")


QUERY 2: TOP 10 HIGHEST CRIME LGAs

SQL Query:

SELECT 
    lga_name,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT year) as years_of_data,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MAX(rate_per_100k), 2) as max_crime_rate
FROM crimes
GROUP BY lga_name
ORDER BY total_incidents DESC
LIMIT 10


What this does:
- Calculates TOTAL incidents for each LGA (summed across all years)
- Shows how many years of data we have
- Shows average and maximum crime rates
- LIMIT 10: Only show top 10 LGAs

Results (Top 10 Highest Crime LGAs):
         lga_name  total_incidents  years_of_data  avg_crime_rate  max_crime_rate
        MELBOURNE           269519             10        16634.30        19331.12
            CASEY           165273             10         4701.71         5823.09
             HUME           150004             10         6444.27         8528.26
GREATER DANDENONG           140944             10         8663.16         9704.40
         BRIMBANK           

We're finding which LGAs have the LEAST crime (safest suburbs). This is the opposite of Query 2.

In [4]:
print("\n" + "="*60)
print("QUERY 3: TOP 10 SAFEST LGAs")
print("="*60)
print("\nSQL Query:")
print("""
SELECT 
    lga_name,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT year) as years_of_data,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MIN(rate_per_100k), 2) as min_crime_rate
FROM crimes
GROUP BY lga_name
ORDER BY total_incidents ASC
LIMIT 10
""")

print("\nWhat this does:")
print("- Calculates TOTAL incidents for each LGA")
print("- Shows minimum crime rate (lowest point)")
print("- ORDER BY total_incidents ASC: Sorts from LEAST to MOST")
print("- LIMIT 10: Show top 10 safest LGAs\n")

# ── RUN THE QUERY ─────────────────────────────────────────────────────────
query3 = """
SELECT 
    lga_name,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT year) as years_of_data,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MIN(rate_per_100k), 2) as min_crime_rate
FROM crimes
GROUP BY lga_name
ORDER BY total_incidents ASC
LIMIT 10
"""

df_query3 = pd.read_sql_query(query3, conn)

print("Results (Top 10 Safest LGAs):")
print(df_query3.to_string(index=False))

# Export to CSV
output_file3 = SQL_OUTPUT + "03_top10_safest.csv"
df_query3.to_csv(output_file3, index=False)
print(f"\n✓ Exported to: {output_file3}")


QUERY 3: TOP 10 SAFEST LGAs

SQL Query:

SELECT 
    lga_name,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT year) as years_of_data,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MIN(rate_per_100k), 2) as min_crime_rate
FROM crimes
GROUP BY lga_name
ORDER BY total_incidents ASC
LIMIT 10


What this does:
- Calculates TOTAL incidents for each LGA
- Shows minimum crime rate (lowest point)
- ORDER BY total_incidents ASC: Sorts from LEAST to MOST
- LIMIT 10: Show top 10 safest LGAs

Results (Top 10 Safest LGAs):
    lga_name  total_incidents  years_of_data  avg_crime_rate  min_crime_rate
   NILLUMBIK            16462             10         2575.42         1866.30
  MANNINGHAM            35213             10         2791.87         2249.35
     BAYSIDE            39199             10         3770.88         3246.42
 HOBSONS BAY            49636             10         5288.89         4712.53
    CARDINIA            50206             10         4484.73         358

We're comparing crime in 2015 (oldest data) vs 2024 (newest data) to see if crime has improved or worsened.

This helps us show:

Which LGAs got safer?
Which got more dangerous?
What's the overall trend?

In [5]:
print("\n" + "="*60)
print("QUERY 4: CRIME COMPARISON 2015 vs 2024")
print("="*60)
print("\nSQL Query:")
print("""
SELECT 
    lga_name,
    MAX(CASE WHEN year = 2015 THEN incidents END) as incidents_2015,
    MAX(CASE WHEN year = 2024 THEN incidents END) as incidents_2024,
    MAX(CASE WHEN year = 2024 THEN incidents END) - 
    MAX(CASE WHEN year = 2015 THEN incidents END) as change,
    ROUND(((MAX(CASE WHEN year = 2024 THEN incidents END) - 
            MAX(CASE WHEN year = 2015 THEN incidents END)) / 
           MAX(CASE WHEN year = 2015 THEN incidents END)) * 100, 2) as pct_change
FROM crimes
WHERE year IN (2015, 2024)
GROUP BY lga_name
ORDER BY pct_change DESC
""")

print("\nWhat this does:")
print("- CASE WHEN: Extracts data for 2015 and 2024 separately")
print("- change: Difference between 2024 and 2015 (positive = more crime)")
print("- pct_change: Percentage change year-over-year")
print("- ORDER BY pct_change DESC: Worst to best\n")

# ── RUN THE QUERY ─────────────────────────────────────────────────────────
query4 = """
SELECT 
    lga_name,
    MAX(CASE WHEN year = 2015 THEN incidents END) as incidents_2015,
    MAX(CASE WHEN year = 2024 THEN incidents END) as incidents_2024,
    MAX(CASE WHEN year = 2024 THEN incidents END) - 
    MAX(CASE WHEN year = 2015 THEN incidents END) as change,
    ROUND(((MAX(CASE WHEN year = 2024 THEN incidents END) - 
            MAX(CASE WHEN year = 2015 THEN incidents END)) / 
           MAX(CASE WHEN year = 2015 THEN incidents END)) * 100, 2) as pct_change
FROM crimes
WHERE year IN (2015, 2024)
GROUP BY lga_name
ORDER BY pct_change DESC
"""

df_query4 = pd.read_sql_query(query4, conn)

print("Results (Crime Change 2015 vs 2024):")
print(df_query4.to_string(index=False))

print(f"\n✓ Positive change = More crime in 2024")
print(f"✓ Negative change = Less crime in 2024 (safer)")

# Export to CSV
output_file4 = SQL_OUTPUT + "04_2015_vs_2024_comparison.csv"
df_query4.to_csv(output_file4, index=False)
print(f"\n✓ Exported to: {output_file4}")


QUERY 4: CRIME COMPARISON 2015 vs 2024

SQL Query:

SELECT 
    lga_name,
    MAX(CASE WHEN year = 2015 THEN incidents END) as incidents_2015,
    MAX(CASE WHEN year = 2024 THEN incidents END) as incidents_2024,
    MAX(CASE WHEN year = 2024 THEN incidents END) - 
    MAX(CASE WHEN year = 2015 THEN incidents END) as change,
    ROUND(((MAX(CASE WHEN year = 2024 THEN incidents END) - 
            MAX(CASE WHEN year = 2015 THEN incidents END)) / 
           MAX(CASE WHEN year = 2015 THEN incidents END)) * 100, 2) as pct_change
FROM crimes
WHERE year IN (2015, 2024)
GROUP BY lga_name
ORDER BY pct_change DESC


What this does:
- CASE WHEN: Extracts data for 2015 and 2024 separately
- change: Difference between 2024 and 2015 (positive = more crime)
- pct_change: Percentage change year-over-year
- ORDER BY pct_change DESC: Worst to best

Results (Crime Change 2015 vs 2024):
            lga_name  incidents_2015  incidents_2024  change  pct_change
        YARRA RANGES            5442         

What We're Doing:

We're looking at Melbourne's OVERALL crime trend from 2015-2024. This answers:

Is Melbourne getting safer or more dangerous?
What year was the safest?
What year had the most crime?

In [6]:
print("\n" + "="*60)
print("QUERY 5: MELBOURNE-WIDE ANNUAL CRIME RATE TREND")
print("="*60)
print("\nSQL Query:")
print("""
SELECT 
    year,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT lga_name) as num_lgas,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MIN(rate_per_100k), 2) as safest_lga_rate,
    ROUND(MAX(rate_per_100k), 2) as highest_crime_rate
FROM crimes
GROUP BY year
ORDER BY year ASC
""")

print("\nWhat this does:")
print("- SUM(incidents): Total crime across ALL Melbourne LGAs for each year")
print("- COUNT(DISTINCT lga_name): How many LGAs have data")
print("- AVG, MIN, MAX: Statistics for the crime rate")
print("- ORDER BY year ASC: From oldest to newest year\n")

# ── RUN THE QUERY ─────────────────────────────────────────────────────────
query5 = """
SELECT 
    year,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT lga_name) as num_lgas,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MIN(rate_per_100k), 2) as safest_lga_rate,
    ROUND(MAX(rate_per_100k), 2) as highest_crime_rate
FROM crimes
GROUP BY year
ORDER BY year ASC
"""

df_query5 = pd.read_sql_query(query5, conn)

print("Results (Melbourne-wide Crime Trend 2015-2024):")
print(df_query5.to_string(index=False))

# Calculate trend
first_year_incidents = df_query5[df_query5['year'] == 2015]['total_incidents'].values[0]
last_year_incidents = df_query5[df_query5['year'] == 2024]['total_incidents'].values[0]
trend = ((last_year_incidents - first_year_incidents) / first_year_incidents) * 100

print(f"\n✓ Overall trend (2015-2024): {trend:+.2f}%")
if trend > 0:
    print(f"  ⚠️ Melbourne got MORE dangerous (crime increased)")
else:
    print(f"  ✅ Melbourne got SAFER (crime decreased)")

# Export to CSV
output_file5 = SQL_OUTPUT + "05_annual_trend_melbourne_wide.csv"
df_query5.to_csv(output_file5, index=False)
print(f"\n✓ Exported to: {output_file5}")


QUERY 5: MELBOURNE-WIDE ANNUAL CRIME RATE TREND

SQL Query:

SELECT 
    year,
    SUM(incidents) as total_incidents,
    COUNT(DISTINCT lga_name) as num_lgas,
    ROUND(AVG(rate_per_100k), 2) as avg_crime_rate,
    ROUND(MIN(rate_per_100k), 2) as safest_lga_rate,
    ROUND(MAX(rate_per_100k), 2) as highest_crime_rate
FROM crimes
GROUP BY year
ORDER BY year ASC


What this does:
- SUM(incidents): Total crime across ALL Melbourne LGAs for each year
- COUNT(DISTINCT lga_name): How many LGAs have data
- AVG, MIN, MAX: Statistics for the crime rate
- ORDER BY year ASC: From oldest to newest year

Results (Melbourne-wide Crime Trend 2015-2024):
 year  total_incidents  num_lgas  avg_crime_rate  safest_lga_rate  highest_crime_rate
 2015           254042        29         6174.52          2775.18            18160.76
 2016           288919        29         6778.09          3225.91            19331.12
 2017           257929        29         5937.64          2875.15            17000.31
 2018  

What We're Doing:

We're checking that all 5 CSV files were created correctly.

In [7]:
print("\n" + "="*60)
print("✅ PHASE 2 COMPLETE - SQL ANALYSIS")
print("="*60)

print(f"\n📁 All exported CSV files:\n")

# Check all files
files_to_check = [
    "01_trend_by_lga_year.csv",
    "02_top10_highest_crime.csv",
    "03_top10_safest.csv",
    "04_2015_vs_2024_comparison.csv",
    "05_annual_trend_melbourne_wide.csv"
]

for i, filename in enumerate(files_to_check, 1):
    filepath = SQL_OUTPUT + filename
    if os.path.exists(filepath):
        filesize = os.path.getsize(filepath) / 1024  # Size in KB
        print(f"✓ {i}. {filename:<45} ({filesize:.2f} KB)")
    else:
        print(f"❌ {i}. {filename:<45} NOT FOUND")

print(f"\n✅ Summary of SQL Analysis:")
print(f"   - Created SQLite database: crime_data.db")
print(f"   - Created 'crimes' table with {len(crime_df)} rows")
print(f"   - Ran 5 SQL queries")
print(f"   - Exported 5 CSV files for Tableau")

print(f"\n📊 Ready for Phase 3: Tableau Dashboard!")

# Close the database connection
conn.close()
print(f"\n✓ Database connection closed")


✅ PHASE 2 COMPLETE - SQL ANALYSIS

📁 All exported CSV files:

✓ 1. 01_trend_by_lga_year.csv                      (8.11 KB)
✓ 2. 02_top10_highest_crime.csv                    (0.41 KB)
✓ 3. 03_top10_safest.csv                           (0.41 KB)
✓ 4. 04_2015_vs_2024_comparison.csv                (0.90 KB)
✓ 5. 05_annual_trend_melbourne_wide.csv            (0.47 KB)

✅ Summary of SQL Analysis:
   - Created SQLite database: crime_data.db
   - Created 'crimes' table with 290 rows
   - Ran 5 SQL queries
   - Exported 5 CSV files for Tableau

📊 Ready for Phase 3: Tableau Dashboard!

✓ Database connection closed
